# 따릉이 시계열 EDA — tsmode (시간별)

- 목적: 집계·변환 **전**, 시간별 원본의 움직임을 파악
- 도구: fg-data-profiling (tsmode) + 직접 그리기 (statsmodels)
- 대상: 서울 공공자전거 **시간별 원본** (8,760행 = 365일 x 24시간)
- 세 가지 관점 (강의자료 66p)
    - 추이: 어디로 가고 있는가 (계절)
    - 주기: 반복되는 패턴 (시간대 · 요일)
    - 이상: 언제 튀었는가 / 빠진 시점

- 왜 시간별인가
    - 따릉이의 가장 강한 패턴은 **시간대별 이봉 분포**다 (31p, 출퇴근 두 번 피크)
    - 일별로 집계하면 이 하루 안의 리듬이 사라진다
    - 그래서 EDA는 시간별 원본으로 먼저 본 뒤, 예측용으로 집계한다

## 1. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_profiling import ProfileReport

## 2. 데이터 불러오기 · timestamp 만들기

- 원본은 Date(날짜)와 Hour(0~23)가 **별도 컬럼**이다
- tsmode·시계열 분석을 하려면 둘을 합쳐 timestamp 하나로 만든다

In [ ]:
df = pd.read_csv("../SeoulBikeData.csv", encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

# Date + Hour -> timestamp
df["timestamp"] = df["Date"] + pd.to_timedelta(df["Hour"], unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)

print(df.shape)          # (8760, ...)
df[["timestamp", "Rented Bike Count", "Temperature(°C)"]].head()

## 3. tsmode 리포트 생성

- 정형 EDA와의 차이: `tsmode=True`, `sortby` 두 줄뿐
- 시간별 원본(8,760행)에 그대로 적용
- 결과: 시간축 그래프 + 자기상관 + 계절성 탐지가 추가됨

In [ ]:
profile = ProfileReport(
    df,
    tsmode=True,             # 시계열 모드
    sortby="timestamp",      # 시간 기준 정렬
    progress_bar=False,
)
profile.to_file("ts_eda.html")   # 브라우저에서 열기

In [ ]:
profile.to_notebook_iframe()

In [ ]:
print(profile.description_set.variables["Rented Bike Count"].keys())

## 4. 리포트에서 확인할 것

- **추이 (시간축 그래프)**
    - 여름 높고 겨울 낮은 계절 패턴이 보이는가
- **주기 (자기상관 · 계절성)**
    - 자기상관 = 과거 값이 미래와 얼마나 닮았는가
    - 하루 24시간 주기, 요일 주기가 잡히는가
- **이상 · 결측**
    - 값이 급변한 구간이 있는가 / 빠진 시점(gap)이 있는가

- 참고: tsmode 파라미터명은 패키지 버전에 따라 다를 수 있다.
  오류 시 `help(ProfileReport)`로 확인할 것.

## 5. 직접 그려보기 — 세 관점을 그래프로

리포트가 자동으로 그려주지만, 핵심은 직접 그려보면 이해가 빠르다.
아래 그래프들이 추이 · 주기 · 이상을 각각 보여준다.

### 5-1. [주기] 시간대별 평균 — 이봉 분포 (따릉이의 핵심)

- 강의자료 31p: 오전·오후 두 번 피크
- 이것이 **일별 집계하면 사라지는** 패턴이다

In [ ]:
ts = df.set_index("timestamp")["Rented Bike Count"]

ts.groupby(ts.index.hour).mean().plot(
    kind="bar", figsize=(10, 3), title="mean by hour of day")
plt.xlabel("hour")
plt.show()

# 확인: 오전/오후 두 번 피크 (출퇴근)

### 5-2. [주기] 요일별 평균 — 주간 주기

In [ ]:
ts.groupby(ts.index.dayofweek).mean().plot(
    kind="bar", figsize=(8, 3), title="mean by weekday (0=Mon)")
plt.xlabel("weekday")
plt.show()

### 5-3. [추이] 전체 흐름 — 계절 패턴

- 시간별은 촘촘해서 계절성이 묻힌다 → 일별 평균으로 그려 추세를 본다

In [ ]:
ts.resample("D").mean().plot(
    figsize=(12, 3), title="daily mean - full period")
plt.ylabel("rentals")
plt.show()

# 확인: 여름 높음 / 겨울 낮음

### 5-4. [이상] 급변 구간 찾기 

In [ ]:
# 일별 평균 기준, 전날 대비 변화가 큰 날
daily_mean = ts.resample("D").mean()
change = daily_mean.diff().abs()
print("변화가 큰 상위 5일:")
print(change.sort_values(ascending=False).head())

# 그날 무슨 일이 있었는가? (예: 폭설/폭우)
# 강의자료 71p 예: 2018-11-24 폭설로 대여량 급락

### 5-5. [이상] 빠진 시점 확인 


In [ ]:
full = pd.date_range(df["timestamp"].min(),
                     df["timestamp"].max(), freq="h")
missing = full.difference(df["timestamp"])
print("전체 시점:", len(full))
print("실제 시점:", len(df))
print("빠진 시점:", len(missing))
# 빠진 시점이 있으면 형식 변환 단계에서 convert_frequency로 채운다

### 5-6. [분포] 치우침 확인

In [ ]:
ts.hist(bins=50, figsize=(8, 3))
plt.title("distribution of hourly rentals")
plt.show()

# 확인: 0 근처에 몰려 있는가 (야간·비운영 시간)
#       오른쪽으로 치우쳤다면 로그 변환 검토

## 정리

- 시계열 EDA = 시간 위의 움직임 (추이 · 주기 · 이상)
- 도구는 정형 EDA와 같다 — tsmode=True 만 추가
- **시간별 원본**으로 봐야 따릉이의 핵심(시간대 이봉)이 보인다
- 자동 리포트(tsmode)
- 자기상관·계절 분해로 계절성을 눈으로 확인
- 여기서 본 것이 예측 단위·모델 선택의 근거가 된다